# Hito 2 — Kernels CUDA (reducción) en Colab
Grupo Laura & Gabriel · Programación Concurrente y Paralela

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)* y guardar.
Ejecutar las celdas **en orden**.

## 1. Verificar la GPU

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Bajar el repo (rama del Hito 2)
Si la rama ya se mezcló en `main`, cambiar `RAMA` por `main`.

In [ ]:
RAMA = "laura-hito2"
!rm -rf de-un-hilo-a-un-llm
!git clone -b {RAMA} https://github.com/Laura-gim/de-un-hilo-a-un-llm.git
%cd de-un-hilo-a-un-llm/hito2-cuda
!ls

## 3. Comprobar que Numba ve la GPU
Si esta celda da error de importación o `False`, ejecutar la celda de ayuda de abajo y reiniciar el entorno.

In [ ]:
from numba import cuda
print("CUDA disponible:", cuda.is_available())
cuda.detect()

In [ ]:
# Solo si la celda anterior falla (el soporte CUDA de Numba puede venir en un paquete aparte):
# !pip install -q numba-cuda
# Luego: Entorno de ejecución -> Reiniciar sesión, y volver a ejecutar desde la celda 2.

## 4. Probar cada kernel por separado
El resultado tiene que ser `Checksum Válido: SÍ` (suma = N).

In [ ]:
!python kernel_ingenuo.py 16777216
!python kernel_optimizado.py 16777216

## 5. Benchmark completo (genera `resultados_gpu.csv` y `resultados_threads.csv`)

In [ ]:
!python benchmark_gpu.py --reps 9 | tee resultados_gpu.txt

## 6. Gráfico

In [ ]:
!pip install -q matplotlib
!python graficar_gpu.py resultados_gpu.csv speedup_gpu.png
from IPython.display import Image
Image("speedup_gpu.png")

## 7. Experimento extra (precisión de float32)
Hipótesis a verificar: con `N > 2^24` el acumulador `float32` del kernel ingenuo **satura en 16 777 216** y el resultado es incorrecto, mientras que el optimizado sigue correcto (suma parcial por bloque y total en `float64`).

In [ ]:
!python kernel_ingenuo.py 33554432
!python kernel_optimizado.py 33554432

## 8. Perfilado con Nsight Compute (si está disponible)
Si `ncu` no existe o falla, **pegar el error en el informe** y usar los tiempos de `cuda.event` (limitación declarada).

In [ ]:
!ncu --version
# Si existe, perfilar (puede tardar):
# !ncu --set full -o ingenuo    python kernel_ingenuo.py 16777216
# !ncu --set full -o optimizado python kernel_optimizado.py 16777216

## 9. Descargar los resultados
Guardarlos en el repo (`hito2-cuda/`) apenas se obtienen: la sesión de Colab se borra.

In [ ]:
from google.colab import files
for f in ["resultados_gpu.txt", "resultados_gpu.csv", "resultados_threads.csv", "speedup_gpu.png"]:
    files.download(f)